# Schema Design 
Milvus on Zilliz Cloud

### 1 - Install dependencies

In [9]:
!pip install --quiet -U pymilvus sentence-transformers


### 2 - Cluster endpoint

In [ ]:
# Paste from Zilliz console (Cluster → Connect → Public Endpoint)
ZILLIZ_URI = ""

if not ZILLIZ_URI:
    raise ValueError("Set ZILLIZ_URI to your Zilliz Public Endpoint before continuing.")

print("URI set:", ZILLIZ_URI)

### 3 - Credentials


In [ ]:
ZILLIZ_USER = ""
ZILLIZ_PASSWORD = ""

if not ZILLIZ_USER or not ZILLIZ_PASSWORD:
    raise ValueError("Set ZILLIZ_USER and ZILLIZ_PASSWORD before continuing.")

print("User set:", ZILLIZ_USER)


### 4 - Connect to Zilliz Cloud

In [ ]:
from pymilvus import MilvusClient

token = f"{ZILLIZ_USER}:{ZILLIZ_PASSWORD}"
client = MilvusClient(uri=ZILLIZ_URI, token=token)

print("Connected to Zilliz Cloud")
print("URI:", ZILLIZ_URI)
print("User:", ZILLIZ_USER)


### 5 - Check Connectivity

In [17]:
collections = client.list_collections()
print(f"Collections ({len(collections)}):")
for name in collections:
    print(f"  - {name}")

if not collections:
    print("(empty cluster — ready to create a schema)")

Collections (1):
  - rag_chunks


### 6. Embedding model (source of truth for dim)

Aligned with `rag_playground_milvus.ipynb`: **`all-MiniLM-L6-v2`**.  
`VECTOR_DIM` is read from the model (not hard-coded).


In [19]:
import os

# Optional: set via env var to avoid Colab HF_TOKEN vault warnings.
# Public models like all-MiniLM-L6-v2 usually work without it.
# Never hardcode a real token here — read from the environment only.
HF_TOKEN = os.getenv("HF_TOKEN", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e:
        print(f"HF login skipped: {e}")
else:
    print("HF_TOKEN not set — public model download may still work")

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # same as chunking / playground

embedder = SentenceTransformer(EMBEDDING_MODEL)

VECTOR_DIM = embedder.get_sentence_embedding_dimension()
MAX_SEQ_LENGTH = embedder.max_seq_length

print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")
print(f"VECTOR_DIM (get_sentence_embedding_dimension): {VECTOR_DIM}")
print(f"MAX_SEQ_LENGTH (embedder.max_seq_length):     {MAX_SEQ_LENGTH}")
try:
    print(f"model.config.hidden_size:                    {embedder[0].auto_model.config.hidden_size}")
except Exception as e:
    print(f"model.config.hidden_size:                    (unavailable: {e})")

print()
print("Note: token CHUNK_SIZE in the chunking notebook is set to MAX_SEQ_LENGTH")
print("so embedded chunks are not truncated by the model.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

EMBEDDING_MODEL: sentence-transformers/all-MiniLM-L6-v2
VECTOR_DIM (get_sentence_embedding_dimension): 384
MAX_SEQ_LENGTH (embedder.max_seq_length):     256
model.config.hidden_size:                    384

Note: token CHUNK_SIZE in the chunking notebook is set to MAX_SEQ_LENGTH
so embedded chunks are not truncated by the model.


/tmp/ipykernel_3024/3324202085.py:26: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_DIM = embedder.get_sentence_embedding_dimension()


### 7. Create schema

Uses `VECTOR_DIM` from the embedding model above.

**If `rag_chunks` already exists (your case): skip this cell.**
Set `OVERWRITE = True` only when you intentionally want to drop and recreate.


In [20]:
from pymilvus import DataType

COLLECTION_NAME = "rag_chunks"
OVERWRITE = False  # True only when recreating (drops data). Schema already exists → skip this cell.

schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=False)

schema.add_field(field_name="chunk_id", datatype=DataType.VARCHAR, is_primary=True, max_length=36, description="Chunk Id")
schema.add_field(field_name="document_id", datatype=DataType.VARCHAR, max_length=36, description="Document Id")
schema.add_field(field_name="chunk_order", datatype=DataType.INT32, description="Chunk order number helps in reconstructing document sequence")
schema.add_field(field_name="base_url", datatype=DataType.VARCHAR, max_length=512, description="Base url of the source website")
schema.add_field(field_name="canonical_url", datatype=DataType.VARCHAR, max_length=512, description="Current url of the document")
schema.add_field(field_name="crawl_date", datatype=DataType.INT64, description="Date when the document was fetched")
schema.add_field(field_name="doc_last_modified", datatype=DataType.INT64, description="Document Last Modified")
schema.add_field(field_name="content_type", datatype=DataType.VARCHAR, max_length=20, description="Type of data whether it is text/image/mixed")
schema.add_field(field_name="content_source_type", datatype=DataType.VARCHAR, max_length=50, description="Type of source e.g., webpage, PDF, announcement")
schema.add_field(field_name="scheme_type", datatype=DataType.VARCHAR, max_length=50, description="Whether it is govt scheme or angel investor etc")
schema.add_field(field_name="scheme_name", datatype=DataType.VARCHAR, max_length=50, description="Name of govt scheme or investor")
schema.add_field(field_name="language", datatype=DataType.VARCHAR, max_length=15, description="Language of the document")
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=15000, description="Chunk original text")
schema.add_field(field_name="text_vector", datatype=DataType.FLOAT_VECTOR, dim=VECTOR_DIM, description="Chunk embedding vector")
schema.add_field(field_name="doc_version", datatype=DataType.VARCHAR, max_length=5, description="Version of the document")
schema.add_field(field_name="is_active", datatype=DataType.BOOL, description="Flag to mark if the document is active")

print(f"Creating collection with text_vector dim={VECTOR_DIM}")

if client.has_collection(COLLECTION_NAME):
    if OVERWRITE:
        client.drop_collection(COLLECTION_NAME)
        print(f"Dropped existing collection: {COLLECTION_NAME}")
    else:
        raise RuntimeError(f"Collection already exists: {COLLECTION_NAME}")

client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
)

print(client.has_collection(COLLECTION_NAME))
print(client.list_collections())


Creating collection with text_vector dim=384
Dropped existing collection: rag_chunks
True
['rag_chunks']


### 8. Load `selected_chunks.json` → `records`

Upload the file from the chunking notebook (Colab Files sidebar, or from Google Drive).

Then run the next cell so `records` is ready for embed + insert.


In [ ]:
import json
from pathlib import Path

# Colab: upload selected_chunks.json to /content, or set a Drive path after mount.
# Drive example (after drive.mount):
# RECORDS_FILE = Path("/content/drive/MyDrive/rag/output/selected_chunks.json")
RECORDS_FILE = Path("selected_chunks.json")

if not RECORDS_FILE.is_file():
    try:
        from google.colab import files
        print("selected_chunks.json not found — pick it from your Mac/Drive download")
        uploaded = files.upload()  # choose selected_chunks.json
        RECORDS_FILE = Path("selected_chunks.json")
        if not RECORDS_FILE.is_file() and uploaded:
            # files.upload writes into cwd under the uploaded name
            name = next(iter(uploaded))
            RECORDS_FILE = Path(name)
    except ImportError:
        raise FileNotFoundError(
            f"Missing {RECORDS_FILE.resolve()}. "
            "Copy it from chunking notebook / Drive into this cwd."
        )

records = json.loads(RECORDS_FILE.read_text(encoding="utf-8"))
assert isinstance(records, list) and records, "JSON must be a non-empty list of chunk dicts"
assert "text" in records[0], "records need a text field"
assert "text_vector" not in records[0] or records[0].get("text_vector") in (None, []), (
    "expected scalars-only JSON; embed in the next section"
)

COLLECTION_NAME = "rag_chunks"
print(f"Loaded {len(records)} records from {RECORDS_FILE.resolve()}")
print("Keys:", list(records[0].keys()))
print("Sample document_id:", records[0].get("document_id"))
print("Collection target:", COLLECTION_NAME)
if not client.has_collection(COLLECTION_NAME):
    raise RuntimeError(f"Collection missing: {COLLECTION_NAME}. Create schema first (§7).")
print("Collection exists:", COLLECTION_NAME)


### 9. Embed `text` → `text_vector`

Uses the same `embedder` / `VECTOR_DIM` from §6.


In [ ]:
texts = [r["text"] for r in records]
vectors = embedder.encode(texts, show_progress_bar=True)

# Ensure plain Python lists of floats with the expected dim
for r, vec in zip(records, vectors):
    v = vec.tolist() if hasattr(vec, "tolist") else list(vec)
    if len(v) != VECTOR_DIM:
        raise ValueError(f"Embedding dim {len(v)} != VECTOR_DIM {VECTOR_DIM}")
    r["text_vector"] = v

print(f"Embedded {len(records)} records")
print(f"text_vector length: {len(records[0]['text_vector'])} (expected {VECTOR_DIM})")
print(f"Keys now: {list(records[0].keys())}")


### 10. Insert into Milvus


In [ ]:
res = client.insert(collection_name=COLLECTION_NAME, data=records)
print("Insert result:", res)
print(f"Inserted {len(records)} rows into {COLLECTION_NAME}")


### 11. Confirm rows are in the collection


In [ ]:
# Peek a few rows (requires vector index + load — run §12 Index first if this fails)
client.load_collection(COLLECTION_NAME)
print("Collection loaded:", COLLECTION_NAME)

rows = client.query(
    collection_name=COLLECTION_NAME,
    filter="chunk_order >= 0",
    output_fields=["chunk_id", "document_id", "chunk_order", "canonical_url", "text"],
    limit=min(5, len(records)),
)
print(f"Query returned {len(rows)} rows")
for row in rows:
    text = (row.get("text") or "")[:120].replace("\n", " ")
    print(f"- {row.get('document_id')} #{row.get('chunk_order')}: {text}...")


### 12. Create vector index on `text_vector`

Required on Zilliz before `load_collection` / query / search.

Uses `AUTOINDEX` + `COSINE` (recommended for Zilliz Cloud).
Safe to re-run: skips if an index already exists.

After this, re-run §11 peek if it failed earlier.


In [22]:
# --- Create vector index (last setup step for query/search) ---
COLLECTION_NAME = "rag_chunks"

existing = client.list_indexes(COLLECTION_NAME)
print("Existing indexes:", existing)

if not existing:
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="text_vector",
        index_type="AUTOINDEX",
        metric_type="COSINE",
    )
    client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)
    print("Created AUTOINDEX on text_vector (COSINE)")
else:
    print("Index already present — skipping create")

print("Indexes now:", client.list_indexes(COLLECTION_NAME))


Existing indexes: []
Created AUTOINDEX on text_vector (COSINE)
Indexes now: ['text_vector']
